# 17.1 Transfer learning

A good representation is reusable knowledge: it turns a tiny target task from learning everything into adapting the last few choices.

Representation learning supplies the feature map being reused; optimization supplies the small target update; regularization explains why freezing lowers variance. Here the same frozen-feature head is tested as labels shrink and shift grows.

Save a copy to Drive to edit.

In [ ]:

import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.datasets import load_digits
from sklearn.datasets import load_iris
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(17)

def budget_ladder():
    """Part 17 (learning paradigms): fix a real base dataset, shrink the LABEL budget per rung.

    Returns [(name, X, y, labeled_mask), ...] over the SAME digits data, only the fraction of
    labeled points falls D1->D5. The 'watch it scale' curve is accuracy vs label budget.
    """
    digits = load_digits()
    X = digits.data / 16.0
    y = digits.target
    rng = np.random.default_rng(17)
    rungs = []
    for name, frac in [("D1 100% labels", 1.0), ("D2 50% labels", 0.5), ("D3 20% labels", 0.2), ("D4 5% labels", 0.05), ("D5 2% labels", 0.02)]:
        mask = rng.random(y.shape) < frac
        if mask.sum() < 20:
            idx = rng.choice(len(y), size=20, replace=False)
            mask = np.zeros(len(y), dtype=bool)
            mask[idx] = True
        rungs.append((name, X, y, mask))
    return rungs


def split_labeled_train_test(X, y, labeled_mask, seed=0):
    train_idx, test_idx = train_test_split(
        np.arange(len(y)),
        test_size=0.35,
        random_state=seed,
        stratify=y,
    )
    labeled_idx = train_idx[labeled_mask[train_idx]]
    if len(np.unique(y[labeled_idx])) < len(np.unique(y)):
        rng = np.random.default_rng(seed)
        needed = []
        for cls in np.unique(y):
            choices = train_idx[y[train_idx] == cls]
            needed.append(rng.choice(choices))
        labeled_idx = np.unique(np.concatenate([labeled_idx, np.array(needed)]))
    return labeled_idx, train_idx, test_idx

def fit_logistic_accuracy(X, y, labeled_mask, seed=0):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, labeled_mask, seed=seed)
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X_labeled = scaler.transform(X[labeled_idx])
    X_test = scaler.transform(X[test_idx])
    clf = LogisticRegression(max_iter=1500, C=2.0, solver="lbfgs")
    clf.fit(X_labeled, y[labeled_idx])
    pred = clf.predict(X_test)
    return accuracy_score(y[test_idx], pred), clf, scaler, labeled_idx, test_idx

def ensure_class_budget_mask(y, frac, seed):
    rng = np.random.default_rng(seed)
    mask = np.zeros(len(y), dtype=bool)
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        count = max(1, int(round(frac * len(cls_idx))))
        chosen = rng.choice(cls_idx, size=count, replace=False)
        mask[chosen] = True
    return mask

def two_dimensional_view(X, seed=0):
    if X.shape[1] == 2:
        return X
    return PCA(n_components=2, random_state=seed).fit_transform(StandardScaler().fit_transform(X))

def plot_panel(ax, X, y, title, marked=None):
    Z = two_dimensional_view(X)
    ax.scatter(Z[:, 0], Z[:, 1], c=y, s=12, cmap="tab10", alpha=0.65)
    if marked is not None and len(marked) > 0:
        M = Z[marked]
        ax.scatter(M[:, 0], M[:, 1], facecolors="none", edgecolors="black", s=45, linewidths=1.0)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


## The concept, built once

Transfer freezes $z=f_	heta(x)$ and updates only the head $p=\sigma(w^Tz+b)$. The lesson's worked point proves that adaptation is a small correction rather than relearning the representation.

In [ ]:

def transfer_head_step(z, w, b, y, eta):
    logit = float(np.dot(w, z) + b)
    prob = 1.0 / (1.0 + math.exp(-logit))
    grad_w = (prob - y) * z
    new_w = w - eta * grad_w
    return logit, prob, grad_w, new_w

z = np.array([2.0, 1.0])
w = np.array([1.2, -0.8])
logit, prob, grad_w, new_w = transfer_head_step(z, w, 0.0, 1.0, 0.1)

assert round(logit, 3) == 1.600
assert round(prob, 3) == 0.832
assert np.allclose(np.round(grad_w, 3), [-0.336, -0.168])
assert np.allclose(np.round(new_w, 3), [1.234, -0.783])

print(f"logit={logit:.3f}, p={prob:.3f}")
print("gradient=", np.round(grad_w, 3))
print("updated w=", np.round(new_w, 3))


A transferred encoder is useful only if its feature geometry is target-relevant. We compare a head on source PCA features against a scratch logistic classifier on raw pixels.

In [ ]:

def make_transfer_ladder():
    digits = load_digits()
    X = digits.data / 16.0
    y = (digits.target >= 5).astype(int)
    rng = np.random.default_rng(171)
    rungs = []
    specs = [
        ("D1 100% target labels", 1.0, 0.00),
        ("D2 50% target labels", 0.5, 0.00),
        ("D3 10% target labels", 0.1, 0.05),
        ("D4 2% target labels", 0.02, 0.10),
        ("D5 1% labels + nuisance shift", 0.01, 0.25),
    ]
    for name, frac, shift in specs:
        Xr = X.copy()
        if shift > 0.0:
            nuisance = rng.normal(0.0, shift, size=Xr[:, :16].shape)
            Xr[:, :16] = Xr[:, :16] + nuisance
        mask = ensure_class_budget_mask(y, frac, seed=int(frac * 1000) + 17)
        rungs.append((name, Xr, y, mask, frac, shift))
    return rungs

def transfer_fit_score(X, y, mask, n_components=12):
    labeled_idx, train_idx, test_idx = split_labeled_train_test(X, y, mask, seed=3)
    source_idx = train_idx[~np.isin(train_idx, labeled_idx)]
    if len(source_idx) == 0:
        source_idx = train_idx
    scaler = StandardScaler()
    scaler.fit(X[source_idx])
    pca = PCA(n_components=n_components, random_state=17)
    pca.fit(scaler.transform(X[source_idx]))
    Z_labeled = pca.transform(scaler.transform(X[labeled_idx]))
    Z_test = pca.transform(scaler.transform(X[test_idx]))
    head = LogisticRegression(max_iter=1500, C=2.0)
    head.fit(Z_labeled, y[labeled_idx])
    transfer_acc = accuracy_score(y[test_idx], head.predict(Z_test))
    scratch_acc, _, _, _, _ = fit_logistic_accuracy(X, y, mask, seed=3)
    return transfer_acc, scratch_acc, pca, labeled_idx, test_idx


## The dataset ladder

D1-D5 keep the target rule fixed while the label budget shrinks and nuisance shift appears. D5 is real digits with 1% labels, matching the plan's hard transfer setting.

In [ ]:

transfer_rungs = make_transfer_ladder()

for name, X, y, mask, frac, shift in transfer_rungs:
    print(f"{name:32s} X={X.shape} labels={mask.sum():4d} frac={frac:.2f} shift={shift:.2f}")

sample_X = transfer_rungs[-1][1][:8]
sample_y = transfer_rungs[-1][2][:8]
print("D5 sample labels:", sample_y.tolist())


In [ ]:

transfer_results = []

for name, X, y, mask, frac, shift in transfer_rungs:
    transfer_acc, scratch_acc, pca, labeled_idx, test_idx = transfer_fit_score(X, y, mask)
    transfer_results.append((name, frac, shift, transfer_acc, scratch_acc, X, y, labeled_idx))
    print(f"{name:32s} transfer={transfer_acc:.3f} scratch={scratch_acc:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(14, 5))

for ax, row in zip(axes[0], transfer_results):
    name, frac, shift, transfer_acc, scratch_acc, X, y, labeled_idx = row
    plot_panel(ax, X, y, name.split()[0] + f" acc={transfer_acc:.2f}", marked=labeled_idx[:40])

budgets = [row[1] for row in transfer_results]
transfer_scores = [row[3] for row in transfer_results]
scratch_scores = [row[4] for row in transfer_results]
axes[1, 0].plot(budgets, transfer_scores, marker="o", label="transfer head")
axes[1, 0].plot(budgets, scratch_scores, marker="s", label="scratch")
axes[1, 0].invert_xaxis()
axes[1, 0].set_xlabel("label budget")
axes[1, 0].set_ylabel("target accuracy")
axes[1, 0].set_title("accuracy vs budget/shift")
axes[1, 0].legend()

for ax in axes[1, 1:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Pitfall on D5: negative transfer

If the source representation emphasizes shifted nuisance pixels, the frozen head can inherit the wrong structure. The fix reduces transferred dimensions so the head uses more stable features.

In [ ]:

name, X, y, mask, frac, shift = transfer_rungs[-1]
bad_acc, scratch_acc, _, _, _ = transfer_fit_score(X, y, mask, n_components=32)
fixed_acc, _, _, _, _ = transfer_fit_score(X, y, mask, n_components=8)

print(f"D5 too many transferred features: {bad_acc:.3f}")
print(f"D5 scratch baseline: {scratch_acc:.3f}")
print(f"D5 reduced transferred features: {fixed_acc:.3f}")
assert fixed_acc >= bad_acc - 0.05


## Evaluate it + Practice

- Metric: target accuracy vs label budget/shift, always compared with a no-skill majority or scratch baseline.
- Sanity check: labels must be shuffled or withheld to confirm the method loses its advantage.
- Ablation: turn off the key paradigm idea and verify the metric drops.
- Failure signal: the diagnostic score improves while held-out target accuracy falls.
- Robustness check: repeat with a different seed and inspect the hardest rung.

### Practice 1
Change the budget or shift on D4 and rerun the summary curve.

### Practice 2
Add one ablation that removes the paradigm-specific step.

### Practice 3
Write a one-sentence deployment warning for D5.